# Convexity

In the previous notebook we ran into trouble with a non-convex energy. Different initializations yielded different answers, and no way to tell which one was the global solution. This notebook makes the word *convexity* precise so we can talk about which problems escape that trouble.

Convexity is a property of two things: the *function* we are trying to optimize and the *set* we are optimizing over. We need both to be convex for the optimization problem itself to be convex.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))
sys.path.append(str(pathlib.Path('../render_utility').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from definitions import blue_2, blue_1, red_2, orange_2, green_2, grey_1

## Convex functions

A function $f$ is convex if, for every pair of points $x_1, x_2$ in its domain and every $t \in [0, 1]$,
$$ f(t x_1 + (1 - t) x_2) \le t f(x_1) + (1 - t) f(x_2). $$
This means the line segment connecting any two points on the graph of $f$ lies on or above the graph itself. The graph never bulges above its own line segments.

Let's see this on two functions: a parabola, which is convex, and a double-well, which is not.

In [ ]:
xs = np.linspace(-2, 2, 400)
f_convex = xs**2
f_double = (xs**2 - 1)**2

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Convex parabola: line segment between two arbitrary points always stays above the graph
axes[0].plot(xs, f_convex, color=blue_2, linewidth=5,
             solid_capstyle='round', solid_joinstyle='miter')
x1, x2 = -1.4, 1.4
y1, y2 = x1**2, x2**2
axes[0].plot([x1, x2], [y1, y2], color=red_2, linewidth=5, linestyle='--',
             dash_capstyle='round')
axes[0].scatter([x1, x2], [y1, y2], color=red_2, s=120,
                edgecolor='white', linewidth=2, zorder=6)
axes[0].set_title('$f(x) = x^2$ (convex)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')

# Double well: we pick the two well bottoms so the line segment lies on the x-axis,
# and the bump at x = 0 sits clearly above it
axes[1].plot(xs, f_double, color=blue_2, linewidth=5,
             solid_capstyle='round', solid_joinstyle='miter')
x1, x2 = -1.0, 1.0
y1, y2 = (x1**2 - 1)**2, (x2**2 - 1)**2
axes[1].plot([x1, x2], [y1, y2], color=red_2, linewidth=5, linestyle='--',
             dash_capstyle='round')
axes[1].scatter([x1, x2], [y1, y2], color=red_2, s=120,
                edgecolor='white', linewidth=2, zorder=6)
axes[1].set_title('$f(x) = (x^2 - 1)^2$ (non-convex)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)')

plt.tight_layout()
plt.show()

On the left, the dashed line segment between the two red dots stays above the curve. The function is convex. On the right, the line segment sits on the x-axis (both red dots are at the well bottoms, where $f = 0$), but the curve bumps up to $f(0) = 1$ in the middle. The line segment is *below* the curve there, which is exactly the condition for non-convexity. That bump is what gradient descent gets stuck on.

Some standard convex functions worth recognizing on sight:
- linear functions, $f(x) = c^\top x + d$,
- norms, $f(x) = \|x\|_p$ for $p \ge 1$,
- the exponential, $f(x) = e^{a x}$,
- and powers $f(x) = x^a$ on $x > 0$ for $a \notin (0, 1)$.

## Convex sets

A set $\Pi \subseteq \mathbb{R}^n$ is convex if, for every two points $x_1, x_2 \in \Pi$ and every $t \in [0, 1]$,
$$ t x_1 + (1 - t) x_2 \in \Pi. $$
Equivalently, the line segment between any two points of the set stays inside the set. There are no dents.

We can visualize this with a few 2D shapes.

In [ ]:
from matplotlib.patches import Polygon, Circle

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

triangle = Polygon([[0.1, 0.1], [0.9, 0.1], [0.5, 0.9]], closed=True,
                   facecolor=blue_2, alpha=0.6, edgecolor=blue_2, linewidth=3)
axes[0].add_patch(triangle)
axes[0].set_title('Triangle (convex)')

circle = Circle((0.5, 0.5), 0.4,
                facecolor=blue_2, alpha=0.6, edgecolor=blue_2, linewidth=3)
axes[1].add_patch(circle)
axes[1].set_title('Disk (convex)')

n_points = 5
outer = np.array([[0.5 + 0.45 * np.cos(2 * np.pi * k / n_points - np.pi / 2),
                   0.5 + 0.45 * np.sin(2 * np.pi * k / n_points - np.pi / 2)] for k in range(n_points)])
inner = np.array([[0.5 + 0.18 * np.cos(2 * np.pi * k / n_points - np.pi / 2 + np.pi / n_points),
                   0.5 + 0.18 * np.sin(2 * np.pi * k / n_points - np.pi / 2 + np.pi / n_points)] for k in range(n_points)])
star_pts = np.vstack([outer, inner]).reshape(2, n_points, 2).transpose(1, 0, 2).reshape(-1, 2)
star = Polygon(star_pts, closed=True,
               facecolor=orange_2, alpha=0.6, edgecolor=orange_2, linewidth=3)
axes[2].add_patch(star)
axes[2].set_title('Star (non-convex)')
# Connect two adjacent tips. The line segment cuts across the V-notch between them
# and clearly leaves the star.
p, q = outer[2], outer[3]
axes[2].plot([p[0], q[0]], [p[1], q[1]], color=red_2, linewidth=5,
             linestyle='--', dash_capstyle='round')
axes[2].scatter([p[0], q[0]], [p[1], q[1]], color=red_2, s=120,
                edgecolor='white', linewidth=2, zorder=6)

for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

The triangle and the disk are convex: take any two points inside, the segment between them stays inside. The star is not. The dashed segment connects two adjacent tips and clearly leaves the shape.

Some standard convex sets worth recognizing:
- affine sets and line segments,
- half-planes and hyperplanes,
- ellipsoids,
- cones, in particular the second-order cone,
- intersections of any of the above as intersections of convex sets are convex.

Let's visualize some of these in 2D! Here are an ellipse, a half-plane, and a second-order cone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

theta = np.linspace(0, 2 * np.pi, 200)
ex = 0.5 + 0.4 * np.cos(theta)
ey = 0.5 + 0.22 * np.sin(theta)
axes[0].fill(ex, ey, color=blue_2, alpha=0.6, edgecolor=blue_2, linewidth=3)
axes[0].set_title('Ellipse (convex)')

# Half-plane {(x, y) : x + y >= 0.6}
xx, yy = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
mask = (xx + yy >= 0.6).astype(float)
axes[1].contourf(xx, yy, mask, levels=[0.5, 1.5], colors=[blue_2], alpha=0.6)
ts = np.linspace(0, 1, 50)
axes[1].plot(ts, 0.6 - ts, color=blue_2, linewidth=3,
             solid_capstyle='round', solid_joinstyle='miter')
axes[1].set_title('Half-plane (convex)')

# Second-order cone {(x, t) : |x| <= t}
ts = np.linspace(0, 1, 100)
left_x  = 0.5 - 0.5 * ts
right_x = 0.5 + 0.5 * ts
axes[2].fill_betweenx(ts, left_x, right_x, color=blue_2, alpha=0.6,
                      edgecolor=blue_2, linewidth=3)
axes[2].plot([0, 0.5, 1], [1, 0, 1], color=blue_2, linewidth=3,
             solid_capstyle='round', solid_joinstyle='miter')
axes[2].set_title('Cone (convex)')

for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

Each of these passes the segment test: take any two points inside, the segment between them stays inside. 

Interestingly, the half-plane is the building block of every linear inequality constraint, and the second-order cone is the geometry behind every $\|\cdot\|_2$ constraint we will write in CVXPY. Most of the feasible sets we'll meet later are intersections of pieces like these!

## Why both conditions matter?

A convex problem needs *both* a convex objective and a convex feasible set. Drop either one and the local-equals-global guarantee is gone.

We can sketch this quickly. Take our convex parabola $f(x) = x^2$. If we minimize it over the convex interval $[-1, 1]$, the answer is $0$ at $x = 0$, and we can find it from any starting point. But if we minimize the same parabola over the *non-convex* set $\{-1, 1\}$, which is a discrete two-point set, we lose convexity of the feasible region. Now the answer depends on which of the two points we are closer to. Similarly, a non-convex objective on a convex domain has the same problem.

## Convex hulls

When we have a set, convex or non-convex, the smallest convex set that contains is called its *convex hull*. Convex hulls are useful in their own right, for example, in collision tests, and they are also a recurring tool in convex relaxation techniques. Let's compute one in 2D!

In [ ]:
from scipy.spatial import ConvexHull
import gpytoolbox as gpy

filename = "../data/chatz.png"
poly_list = gpy.png2poly(filename)
points = None
for poly in poly_list:
    p = poly[::50,:]
    points = p if points is None else np.concatenate((points, p), axis=0)
points = gpy.normalize_points(points)
hull = ConvexHull(points)

fig, ax = plt.subplots(figsize=(5, 5))
for simplex in hull.simplices:
    ax.plot(points[simplex, 0], points[simplex, 1], color=blue_2,
            linewidth=5, solid_capstyle='round', solid_joinstyle='miter')
ax.scatter(points[:, 0], points[:, 1], color=grey_1, s=20)
ax.set_aspect('equal'); ax.set_title('Convex hull of a 2D point cloud')
plt.show()

In [ ]:
import polyscope as ps

V, F = gpy.read_mesh('../data/bunny.obj')
V = np.ascontiguousarray(V[:, :3], dtype=np.float64)
points = V[:, :10]
hull = ConvexHull(points)

ps.init()
ps.set_ground_plane_mode('none')      # no floor
ps.set_transparency_mode('pretty')
ps.register_surface_mesh("bunny", V*0.99, F, color=(0.8, 0.8, 0.8)) # shrink a bit to avoid z-fighting
ps.register_surface_mesh("convex hull", points, hull.simplices, transparency=0.6, color=(31.0/255.0,120.0/255.0,180.0/255.0))
ps.show()


## Summary

We now have a working definition of *convexity* for both functions and sets, and we have seen what each looks like in a few examples. In the next notebook we will start using this language in practice: we'll write our first convex program in CVXPY and put the convexity guarantees to the test.

## References

[1] *Convex Optimization*. Boyd, S.; Vandenberghe, L.
       Cambridge University Press (2004).
       :DOI:`10.1017/CBO9780511804441`